# Grid vs. Random vs. Bayesian: The Showdown

*Companion to* **Mastering CatBoost — Pro Edition**, Chapter 4: Tuning & Diagnostics as System Control, §4.1 (Overview) and §4.2 (The Tuning Philosophy).

Chapter 4 opens by calling grid search over "a bag of unrelated knobs" inefficient and dangerous. This notebook puts the claim to the test under controlled conditions: three search strategies, the **same budget** (24 model fits each), the **same data**, the **same search space**, the **same seed discipline**. The notebook computes the comparison — no result is asserted in advance.

In [ ]:
%pip install -q catboost==1.2.10 scikit-learn==1.6.1 numpy==2.0.2 pandas==2.2.2 matplotlib==3.10.0 optuna==4.2.1

import itertools
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from catboost import CatBoostClassifier

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Fixed experimental conditions

- **Evaluation budget:** 24 early-stopped development fits per strategy. Retained trees and wall time may differ.
- **Bounds:** all strategies use the same declared parameter bounds, but grid samples coarse lattice points while random and TPE sample the continuous/log domains.
- **Data:** search on train/validation; freeze each strategy's champion before final evaluation.
- **Comparison:** equal evaluation count is not equal compute. Report both fit count and wall time.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=8_000, n_features=20, n_informative=9,
                           n_redundant=4, flip_y=0.1, random_state=RANDOM_SEED)
X = X.astype(np.float32)
X_tr, X_rest, y_tr, y_rest = train_test_split(X, y, test_size=0.4, random_state=RANDOM_SEED, stratify=y)
X_va, X_te, y_va, y_te = train_test_split(X_rest, y_rest, test_size=0.5, random_state=RANDOM_SEED, stratify=y_rest)

BUDGET = 24

def evaluate(lr, depth, l2):
    m = CatBoostClassifier(iterations=400, learning_rate=lr, depth=int(depth), l2_leaf_reg=l2,
                           eval_metric="AUC", random_seed=RANDOM_SEED, verbose=0,
                           allow_writing_files=False)
    m.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=40, use_best_model=True)
    return m.get_best_score()["validation"]["AUC"], m

print(f"budget: {BUDGET} evaluations per strategy")

## 2. Strategy A — Grid search

A 3 × 4 × 2 lattice = 24 points. Note the built-in dilemma: with a fixed budget, every extra value for one parameter steals resolution from the others, and continuous parameters get only a handful of coarse levels.

In [ ]:
grid_lr = [0.03, 0.1, 0.3]
grid_depth = [4, 6, 8, 10]
grid_l2 = [1.0, 9.0]

grid_trials = []
t0 = time.perf_counter()
for lr, d, l2 in itertools.product(grid_lr, grid_depth, grid_l2):
    auc, _ = evaluate(lr, d, l2)
    grid_trials.append({"lr": lr, "depth": d, "l2": l2, "valid_auc": auc})
grid_time = time.perf_counter() - t0
grid_df = pd.DataFrame(grid_trials)
grid_best = grid_df.loc[grid_df.valid_auc.idxmax()]
print(f"grid: best valid AUC {grid_best.valid_auc:.4f} at lr={grid_best.lr}, "
      f"depth={int(grid_best.depth)}, l2={grid_best.l2} | {grid_time:.0f}s")

## 3. Strategy B — Random search

24 independent draws from the *continuous* space. Random search's classic advantage (Bergstra & Bengio's argument): when only some dimensions matter, random points give 24 distinct values along **every** axis, while the grid gave `learning_rate` just 3.

In [ ]:
rng = np.random.RandomState(RANDOM_SEED)
random_trials = []
t0 = time.perf_counter()
for _ in range(BUDGET):
    lr = float(np.exp(rng.uniform(np.log(0.02), np.log(0.3))))
    d = int(rng.randint(4, 11))
    l2 = float(np.exp(rng.uniform(np.log(0.5), np.log(30))))
    auc, _ = evaluate(lr, d, l2)
    random_trials.append({"lr": lr, "depth": d, "l2": l2, "valid_auc": auc})
random_time = time.perf_counter() - t0
random_df = pd.DataFrame(random_trials)
random_best = random_df.loc[random_df.valid_auc.idxmax()]
print(f"random: best valid AUC {random_best.valid_auc:.4f} at lr={random_best.lr:.3f}, "
      f"depth={int(random_best.depth)}, l2={random_best.l2:.2f} | {random_time:.0f}s")

## 4. Strategy C — Bayesian search (Optuna TPE)

Same space, same budget, but each new point is *proposed* from a model of past results: TPE splits observed trials into good and bad groups and samples where the good density dominates. The first few trials are effectively random (startup), then adaptation kicks in — watch for that in the trajectory plot.

In [ ]:
def objective(trial):
    lr = trial.suggest_float("lr", 0.02, 0.3, log=True)
    d = trial.suggest_int("depth", 4, 10)
    l2 = trial.suggest_float("l2", 0.5, 30.0, log=True)
    auc, _ = evaluate(lr, d, l2)
    return auc

study = optuna.create_study(direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
t0 = time.perf_counter()
study.optimize(objective, n_trials=BUDGET)
tpe_time = time.perf_counter() - t0
tpe_df = study.trials_dataframe()
print(f"TPE: best valid AUC {study.best_value:.4f} at {study.best_params} | {tpe_time:.0f}s")

## 5. The comparison — computed, not asserted

In [ ]:
summary = pd.DataFrame([
    {"strategy": "grid",   "best_valid_auc": round(grid_df.valid_auc.max(), 4),
     "mean_valid_auc": round(grid_df.valid_auc.mean(), 4), "wall_s": round(grid_time)},
    {"strategy": "random", "best_valid_auc": round(random_df.valid_auc.max(), 4),
     "mean_valid_auc": round(random_df.valid_auc.mean(), 4), "wall_s": round(random_time)},
    {"strategy": "TPE",    "best_valid_auc": round(study.best_value, 4),
     "mean_valid_auc": round(tpe_df["value"].mean(), 4), "wall_s": round(tpe_time)},
])
summary

In [ ]:
plt.figure(figsize=(9, 4.5))
for name, vals, color in [("grid", grid_df.valid_auc.values, "tab:gray"),
                          ("random", random_df.valid_auc.values, "tab:blue"),
                          ("TPE", tpe_df["value"].values, "tab:red")]:
    best_so_far = np.maximum.accumulate(vals)
    plt.plot(range(1, len(vals) + 1), best_so_far, "-o", ms=3, label=name, color=color)
plt.xlabel("evaluations used"); plt.ylabel("best validation AUC so far")
plt.title(f"Best-so-far trajectories, equal budget of {BUDGET} fits")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# One-shot test evaluation of each strategy's champion (touch the test set once, at the end).
champions = {
    "grid": (grid_best.lr, grid_best.depth, grid_best.l2),
    "random": (random_best.lr, random_best.depth, random_best.l2),
    "TPE": (study.best_params["lr"], study.best_params["depth"], study.best_params["l2"]),
}
rows = []
for name, (lr, d, l2) in champions.items():
    _, m = evaluate(lr, d, l2)
    rows.append({"strategy": name,
                 "test_AUC": round(roc_auc_score(y_te, m.predict_proba(X_te)[:, 1]), 4)})
pd.DataFrame(rows)

### How to read the results

Compare best-so-far trajectories, selected validation values, total wall time, and repeated search seeds. Mean trial value is not the search objective and need not favor TPE. Grid, random, or TPE can win a finite noisy run; the coarse grid also searches a different discrete support within the shared bounds.

## Takeaways

- Grid allocates evaluations to a declared lattice; random and TPE sample broader continuous domains.
- No method is guaranteed to dominate at low effective dimension or at a stated budget.
- TPE's adaptation can help or hurt in finite runs; pruning and early stopping change wall costs.
- Freeze candidates before one-shot test evaluation and compare margins with seed variability.

## Exercises

**Exercise 1 - Replication.** Repeat all stochastic strategies across search seeds and report the rank distribution.

**Exercise 2 - Budget scaling.** Compare 8, 24, and 48 evaluations and wall-clock budgets. Report live trajectories; do not prescribe widening TPE advantage.

**Exercise 3 - Add a conditional dimension.** Compare supports and compute budgets explicitly. Higher dimensionality motivates alternatives to exhaustive grids but does not guarantee a random/TPE winner.

### Hidden solutions

<details><summary>Solution</summary>

**Exercise 1.** Run each stochastic search for at least ten sampler seeds. For every run store best validation value, selected parameters, evaluations completed, wall time, and rank among methods. Report the win count and paired score differences with intervals; do not use average trial value as the selection criterion.

</details>

<details><summary>Solution</summary>

**Exercise 2.** At budgets 8, 24, and 48, reconstruct best-so-far validation trajectories from trial order and separately plot best-so-far against cumulative wall time. Early stopping makes equal evaluation counts unequal in cost, so provide both views. Compare seed distributions at each budget rather than joining one lucky trajectory.

</details>

<details><summary>Solution</summary>

**Exercise 3.** Add the conditional parameter only on the branch where it is valid, and publish each method's actual support: grid points, random distributions, and TPE distributions. Hold the validation evaluator fixed, compare both evaluation-count and wall-clock budgets, and report coverage of the conditional branch plus selected quality.

</details>
